In [21]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import make_grid
from IPython.display import clear_output

In [22]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
writer=SummaryWriter()
channels=1
batch_size=100
features_size=channels*28*28
z_dim=100

trans=T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.5 for _ in range(channels)],std=[0.5 for _ in range(channels)])
])

data=datasets.MNIST('data',train=True,download=True,transform=trans)

train_loader=DataLoader(data,batch_size=batch_size,shuffle=True,drop_last=True)

In [23]:
class Discriminator(nn.Module):
  def __init__(self,in_features,hidden_features):
    super().__init__()

    self.disc=nn.Sequential(
        nn.Linear(in_features,hidden_features),
        nn.LeakyReLU(),
        nn.Linear(hidden_features,hidden_features),
        nn.LeakyReLU(),
        nn.Linear(hidden_features,1),
        nn.Sigmoid()
    )


  def forward(self,x):
    return self.disc(x)


class Generator(nn.Module):
  def __init__(self,in_features,hidden_features):
    super().__init__()

    self.gen=nn.Sequential(
        nn.Linear(in_features,hidden_features),
        nn.LeakyReLU(),
        nn.Linear(hidden_features,hidden_features),
        nn.LeakyReLU(),
        nn.Linear(hidden_features,784),
        nn.Tanh()
    )

  def forward(self,x):
    return self.gen(x)

In [24]:
disc=Discriminator(features_size,128).to(device)
gen=Generator(z_dim,128).to(device)
real_test=torch.randn(batch_size,channels,28,28).flatten(start_dim=1).to(device)
fake_test=torch.randn(batch_size,z_dim).to(device)

real_disc_test=disc(real_test)
fake_img=gen(fake_test)
fake_disc_test=disc(fake_img)

print(f'''
real_pred size : {real_disc_test.shape}
fake_img size : {fake_img.shape}
fake_pred size : {fake_disc_test.shape}
''')


real_pred size : torch.Size([100, 1])
fake_img size : torch.Size([100, 784])
fake_pred size : torch.Size([100, 1])



In [25]:
lossfun=nn.BCELoss()
lr=0.0005
epochs=200
fixed_noise=torch.randn(batch_size,z_dim)
disc_optimizer=torch.optim.Adam(disc.parameters(),lr=lr)
gen_optimizer=torch.optim.Adam(gen.parameters(),lr=lr)

In [27]:
for epoch in range(epochs):
  disc_running_loss=0
  gen_running_loss=0

  for idx,(real_image,_) in enumerate(train_loader):
    real_image=real_image.view(real_image.size(0),-1).to(device)
    noise=torch.randn(batch_size,z_dim).to(device)
    fake_image=gen(noise)

    # training Discriminator
    real_pred=disc(real_image)
    fake_pred=disc(fake_image.detach())

    real_loss=lossfun(real_pred,torch.ones_like(real_pred))
    fake_loss=lossfun(fake_pred,torch.zeros_like(fake_pred))

    disc_loss=(real_loss+fake_loss)

    disc_optimizer.zero_grad()
    disc_loss.backward()
    disc_optimizer.step()

    # training Generator

    fake_real_pred=disc(fake_image)
    gen_loss=lossfun(fake_real_pred,torch.ones_like(fake_real_pred))

    gen_optimizer.zero_grad()
    gen_loss.backward()
    gen_optimizer.step()

    disc_running_loss+=disc_loss.item()
    gen_running_loss+=gen_loss.item()

  disc_epoch_loss=disc_running_loss/len(train_loader)
  gen_epoch_loss=gen_running_loss/len(train_loader)

  if epoch%1 ==0:
    print(f'{epoch+1}/{epochs} Discriminator Loss : {disc_epoch_loss:.2f} Generator Loss : {gen_epoch_loss:.2f}')

    clear_output(wait=True)

    # with torch.no_grad():
    #   fixed_noise=fixed_noise.to(device)
    #   fake_over_epoch=gen(fixed_noise)[:16]
    #   real_over_epoch=real_image[:16]

    #   fake_reshaped=fake_over_epoch.reshape(-1,1,28,28)
    #   real_reshaped=real_over_epoch.reshape(-1,1,28,28)

    #   fake_grid=make_grid(fake_reshaped,normalize=True)
    #   real_grid=make_grid(real_reshaped,normalize=True)

    #   writer.add_image('Fake_images',fake_grid,global_step=epoch)
    #   writer.add_image('Real_images',real_grid,global_step=epoch)
    #   writer.add_scalar('Generator Loss',gen_epoch_loss,global_step=epoch)
    #   writer.add_scalar('Discriminator Loss',disc_epoch_loss,global_step=epoch)

200/200 Discriminator Loss : 0.98 Generator Loss : 1.44
